In [ ]:
!pip -q install sentence-transformers openpyxl

In [ ]:
import re, glob, os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- 1. Baca semua Excel/CSV di folder ----------
ALIAS = {"chapter1": "latar_belakang", "abstract": "abstrak", "title": "judul", "year": "tahun"}

def muat_data(folder="/content"):
    frames = []
    for p in sorted(glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)):
        if not p.lower().endswith((".xlsx", ".xls", ".csv")):
            continue
        sheets = {"": pd.read_csv(p)} if p.lower().endswith(".csv") else pd.read_excel(p, sheet_name=None)
        for df in sheets.values():
            df.columns = [ALIAS.get(c, c) for c in (str(c).strip().lower().replace(" ", "_") for c in df.columns)]
            if "judul" in df.columns:                     # lewati sheet/file yang bukan data skripsi
                df = df.loc[:, ~df.columns.duplicated()].copy()
                df["jurusan"] = os.path.splitext(os.path.basename(p))[0]
                frames.append(df)
    db = pd.concat(frames, ignore_index=True)
    for k in ("judul", "abstrak", "latar_belakang"):
        db[k] = db[k].fillna("").astype(str).str.strip() if k in db else ""
    db = db[(db.judul.str.upper() != "TIDAK DITEMUKAN") & (db.abstrak.str.len() > 50)]   # buang baris kosong
    return db.drop_duplicates("judul").reset_index(drop=True)

# ---------- 2. Bersihkan teks & potong Bab 1 ----------
STOP = set("""yang dan di ke dari untuk pada dengan dalam adalah ini itu atau juga serta oleh sebagai dapat akan agar
karena sehingga bahwa para tersebut terhadap antara sebuah suatu ada tidak lebih sangat telah secara hal saat masih""".split())

def bersih(t):
    return " ".join(w for w in re.sub(r"[^a-z0-9\s]", " ", str(t).lower()).split() if w not in STOP and len(w) > 1)

NOMOR = re.compile(r"(?<![\d.])1\.\d{1,2}\.?\s+(?=[A-Z])")          # heading bernomor: 1.4 / 1.4.
BAGIAN = {"rumusan": r"(?:rumusan|perumusan|identifikasi)\s+masalah",
          "lingkup": r"ruang\s+lingkup|batasan(?:\s+masalah|\s+penelitian)?"}

def bab1(teks, maks=1500):
    """Ambil 'Rumusan Masalah' dan 'Ruang Lingkup' dari Bab 1. Latar Belakang dibuang."""
    teks = re.sub(r"\d*\s*Universitas Kristen Petra\s*-*", " ", re.sub(r"\s+", " ", teks))
    mulai = list(NOMOR.finditer(teks))
    hasil = dict.fromkeys(BAGIAN, "")
    for i, m in enumerate(mulai):
        akhir = mulai[i + 1].start() if i + 1 < len(mulai) else len(teks)
        for k, pola in BAGIAN.items():
            h = re.match(pola, teks[m.end():m.end() + 60], re.I)
            if h:
                hasil[k] = (hasil[k] + " " + teks[m.end() + h.end():akhir].strip()).strip()
    return hasil["rumusan"][:maks], hasil["lingkup"][:maks]

# ---------- 3. Pencocokan kata kunci (BM25) ----------
class BM25:
    def __init__(s, docs, k1=1.5, b=0.75):
        s.k1, s.b, s.tf = k1, b, [Counter(d) for d in docs]
        s.dl = np.array([len(d) for d in docs], float)
        s.avg = s.dl.mean() if s.dl.mean() > 0 else 1.0
        n, df = len(docs), Counter(t for d in docs for t in set(d))
        s.idf = {t: np.log((n - f + .5) / (f + .5) + 1) for t, f in df.items()}
        s.idf_maks = np.log((n + .5) / .5 + 1)                      # idf untuk kata yang tidak ada di database

    def _s(s, f, dl):
        return f * (s.k1 + 1) / (f + s.k1 * (1 - s.b + s.b * dl / s.avg))

    def skor(s, q):
        return np.array([sum(s.idf[t] * s._s(tf[t], dl) for t in set(q) if t in s.idf and tf[t])
                         for tf, dl in zip(s.tf, s.dl)])

    def acuan(s, q):                                                  # skor maksimum wajar, untuk normalisasi 0-1
        return sum(s.idf.get(t, s.idf_maks) * s._s(f, max(len(q), 1)) for t, f in Counter(q).items())

def sim_tfidf(korpus, q):                                             # cadangan jika SBERT tidak ada
    kb, qb = [bersih(t) for t in korpus], bersih(q)
    if not qb or not any(kb):
        return np.zeros(len(korpus))
    X = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(kb + [qb])
    return cosine_similarity(X[-1], X[:-1]).ravel()

# ---------- 4. Mesin ----------
class Mesin:
    def __init__(self, db, alpha=0.7, bobot=(0.5, 0.5), sbert=True, encoder=None):
        d = db.copy().reset_index(drop=True)
        d["rumusan"], d["lingkup"] = zip(*[bab1(t) for t in d.latar_belakang])
        self.d, self.alpha, self.bobot = d, alpha, bobot
        self.abs = d.judul + ". " + d.abstrak                                         # teks untuk kemiripan makna
        self.ada = (d.lingkup.str.strip() != "").values                              # topik yang punya Ruang Lingkup
        self.bm25 = BM25([bersih(t).split() for t in d.judul + " " + d.abstrak + " " + d.lingkup + " " + d.rumusan])
        self.enc = encoder
        if encoder is None and sbert:
            try:
                from sentence_transformers import SentenceTransformer
                m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
                self.enc = lambda t: m.encode(t, normalize_embeddings=True, show_progress_bar=False, batch_size=32)
            except Exception as e:
                print("SBERT tidak tersedia, memakai TF-IDF:", type(e).__name__)
        self.mirip, self.sangat = (0.60, 0.75) if self.enc else (0.30, 0.50)         # ambang menyesuaikan skala skor
        if self.enc:
            self.E_abs = np.asarray(self.enc(list(self.abs)), float)
            self.E_ling = np.asarray(self.enc([t if t.strip() else " " for t in d.lingkup]), float)

    def skor(self, judul, masalah="", metode="", data="", deskripsi=""):
        gabung = lambda *x: ". ".join(i.strip() for i in x if i and i.strip())
        q = gabung(judul, masalah, metode, data, deskripsi)
        ql = gabung(judul, metode, data) if (metode.strip() or data.strip()) else q   # ruang lingkup: metode + dataset
        if self.enc:
            s_abs = np.clip(self.E_abs @ np.asarray(self.enc([q]), float)[0], 0, 1)
            s_ling = np.clip(self.E_ling @ np.asarray(self.enc([ql]), float)[0], 0, 1)
        else:
            s_abs, s_ling = sim_tfidf(self.abs, q), sim_tfidf(self.d.lingkup, ql)
        sem = np.where(self.ada, self.bobot[0] * s_abs + self.bobot[1] * s_ling, s_abs)
        qt = bersih(q).split()
        a = self.bm25.acuan(qt)
        bm = np.clip(self.bm25.skor(qt) / a, 0, 1) if a > 0 else np.zeros(len(self.d))
        return self.alpha * sem + (1 - self.alpha) * bm                              # skor hybrid

    def tingkat(self, s):
        return "SANGAT MIRIP" if s >= self.sangat else "MIRIP" if s >= self.mirip else "RENDAH"

    def cari(self, judul, masalah="", metode="", data="", top_k=5):
        s = self.skor(judul, masalah, metode, data)
        i = np.argsort(s)[::-1][:top_k]
        return pd.DataFrame({"judul": self.d.judul[i].str[:80].values, "jurusan": self.d.jurusan[i].values,
                             "skor": s[i].round(3), "tingkat": [self.tingkat(x) for x in s[i]]})

In [ ]:
db = muat_data("/content")
print(len(db), "topik valid"); print(db.groupby("jurusan").size())   # cek: DSA 54, INFOR 68, SIB 64
mesin = Mesin(db)                                                     # pertama kali mengunduh model SBERT
print("Mode:", "SBERT" if mesin.enc else "TF-IDF")

777 topik valid
jurusan
DSA       54
INFOR    517
SIB      206
dtype: int64


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Mode: SBERT


In [ ]:
# (id, grup, judul, masalah, metode, dataset, sumber_topik, catatan)
# grup A = hampir identik (parafrase), B = berkaitan tapi berbeda, C = orisinal.
# sumber_topik = awalan judul skripsi di Excel (huruf kecil), dipisah " | " bila lebih dari satu.
DATA = [
 # ===== A. 10 proposal parafrase dari skripsi di Excel =====
 ("A01", "A", "Aplikasi Web Belajar Aksara Sunda dengan Pengenalan Karakter Berbasis EfficientNetV2B0",
  "Aksara Sunda terancam hilang karena generasi muda jarang memakainya dan belum ada media belajar yang interaktif.",
  "Klasifikasi karakter dengan EfficientNetV2B0, deteksi dan segmentasi gambar dengan YOLOv8, serta augmentasi data.",
  "Gambar aksara Sunda (ngalagena, swara, rarangken) dari dataset publik di internet dan tulisan tangan responden.",
  "aplikasi pembelajaran dan pengenalan aksara s", "Parafrase DSA 2025 (aksara Sunda, EfficientNetV2B0)"),
 ("A02", "A", "Peningkatan Akurasi Pengenalan Emosi dari Suara dengan Menggabungkan Teks dan Audio Memakai wav2vec dan SpecAugment",
  "Pengenalan emosi suara yang hanya memakai satu modalitas kurang akurat dalam membedakan marah, sedih, senang, dan netral.",
  "Model multimodal yang menggabungkan teks hasil speech to text dengan sinyal suara, memakai wav2vec dan augmentasi SpecAugment.",
  "Rekaman suara berformat wav dari RAVDESS dan IEMOCAP beserta transkripnya.",
  "integrasi teks dan suara dengan model wav2vec", "Parafrase DSA 2025 (speech emotion recognition)"),
 ("A03", "A", "Faktor yang Memengaruhi Indeks Pembangunan Manusia di Kabupaten dan Kota Jawa Timur dengan Regresi Data Panel dan Random Forest",
  "IPM Jawa Timur terus naik tetapi kesenjangan antarwilayah masih lebar, sehingga perlu diketahui faktor penentunya di tiap daerah.",
  "Regresi data panel, random forest untuk tingkat kepentingan faktor, dan K-Means untuk mengelompokkan wilayah.",
  "Data sekunder BPS untuk 38 kabupaten dan kota di Jawa Timur periode 2017 sampai 2024.",
  "analisis faktor-faktor yang mempengaruhi inde", "Parafrase DSA 2026 (IPM Jawa Timur)"),
 ("A04", "A", "Penentuan Lokasi Terbaik Pembangkit Listrik Tenaga Bayu di Nusa Tenggara Timur dengan Analisis Spasio-Temporal dan Random Forest",
  "Potensi angin di NTT besar, tetapi memilih lokasi pembangkit listrik tenaga angin yang optimal sulit karena karakter angin berubah menurut ruang dan waktu.",
  "Klasifikasi kelayakan lokasi dengan random forest memakai fitur kecepatan, arah, dan variabilitas angin.",
  "Data angin historis NASA POWER tahun 2016 sampai 2025 pada 27 lokasi, termasuk lokasi PLTB yang sudah beroperasi sebagai acuan.",
  "analisis data spasio-temporal untuk identifik", "Parafrase DSA 2026 (lokasi optimal PLTB NTT)"),
 ("A05", "A", "Pembacaan Gerak Bibir Bahasa Indonesia Berdasarkan Visem dengan VGG16, Jaro-Winkler, dan Bigram",
  "Mengenali kata hanya dari gerakan bibir berguna bagi penyandang gangguan pendengaran, tetapi belum banyak model untuk bahasa Indonesia.",
  "Klasifikasi visem dengan VGG16 yang dibandingkan dengan STCNN, lalu koreksi kata memakai kemiripan Jaro-Winkler dan bigram.",
  "Video buatan sendiri berisi 25 kalimat bahasa Indonesia (50 kata) yang diucapkan 12 orang, total 300 video.",
  "pemodelan lip reading bahasa indonesia berbas", "Parafrase INFOR 2022 (lip reading visem)"),
 ("A06", "A", "Kendali Presentasi dan Pemutar Media dengan Gerakan Tangan Memakai Mediapipe untuk Konferensi Hibrida",
  "Pada rapat atau kuliah hibrida, presenter sulit mengoperasikan slide dan pemutar media tanpa menyentuh komputer.",
  "Pengenalan gerakan tangan dinamis dengan CNN 3D (RT3D) dan Mediapipe Hands, lalu gerakan dipetakan ke perintah aplikasi.",
  "Jester dataset dari TwentyBN (video gestur tangan, 27 kelas, 12 dipakai) dan video langsung dari kamera.",
  "deteksi hand gesture dengan menggunakan media", "Parafrase INFOR 2022 (hand gesture Mediapipe)"),
 ("A07", "A", "Aplikasi Web Penerbitan Sertifikat dengan Pemeriksaan Keaslian Memakai SHA-256 dan RSA",
  "Penerbitan sertifikat memerlukan banyak proses, dan keaslian sertifikat digital sulit dipastikan.",
  "Tanda tangan digital dengan hash SHA-256 dan RSA, diimplementasikan di PHP memakai pustaka phpseclib.",
  "Data sertifikat yang dibuat di aplikasi beserta tanda tangan digitalnya.",
  "aplikasi pembuatan sertifikat dengan validasi", "Parafrase INFOR 2023 (sertifikat SHA-256 RSA)"),
 ("A08", "A", "Gudang Data Berbasis Graph Database untuk Menganalisis Pelaksanaan Kurikulum OBE Program Studi Informatika",
  "Prodi perlu menilai apakah kurikulum berbasis outcome berjalan baik, tetapi data CPL, CPMK, dan nilai tersebar sehingga sulit dianalisis.",
  "Data warehouse dengan graph database Neo4j dan analisis distribusi nilai CPL.",
  "Data akademik prodi (mahasiswa, CPL, CPMK, kelas, mata kuliah, nilai) semester gasal 2020 sampai genap 2022.",
  "data warehouse dengan graph database untuk an", "Parafrase SIB 2024 (data warehouse graph database)"),
 ("A09", "A", "Optimasi Rute Pengantaran pada Jasa Kurir dengan Clarke and Wright Saving dan Cheapest Insertion",
  "Jasa pengiriman di Surabaya membagi alamat antar driver secara tidak seimbang sehingga jarak tempuh tidak minimum.",
  "Clarke and Wright Saving Heuristic dan Cheapest Insertion Heuristic, rute divisualisasikan dengan Google Maps API.",
  "Data alamat pengiriman dan kapasitas barang harian dari perusahaan jasa pengiriman di Surabaya.",
  "sistem optimalisasi rute menggunakan metode c", "Parafrase SIB 2023 (rute Clarke and Wright)"),
 ("A10", "A", "Aplikasi Pendamping Konsultasi Diet Online untuk Memantau Member dan Menghitung Kebutuhan Gizi",
  "Coach diet kesulitan memantau banyak member aktif satu per satu pada bisnis konsultasi diet online.",
  "Sistem informasi dengan perhitungan kebutuhan kalori dan komposisi gizi dari profil tubuh member.",
  "Tabel Komposisi Pangan Indonesia (TKPI) dan profil member (usia, tinggi, berat, lingkar tubuh).",
  "aplikasi konsultasi diet jp smart fitness", "Parafrase SIB 2023 (konsultasi diet JP Smart Fitness)"),
 # ===== B. 10 proposal berkaitan tetapi berbeda =====
 ("B01", "B", "Prediksi Hasil Panen Jagung Berdasarkan Curah Hujan dan Indeks Vegetasi Satelit",
  "Petani dan dinas pertanian sulit memperkirakan hasil panen per musim karena cuaca berubah-ubah.",
  "Gradient boosting (LightGBM) dengan fitur curah hujan dan NDVI.",
  "Citra Sentinel-2 untuk NDVI dan data curah hujan CHIRPS di Jawa Timur.",
  "prediksi produktivitas komoditas strategis be", "Inspirasi DSA 2026 (produktivitas pertanian dari tanah dan cuaca). Problem sama, dataset dan metode beda"),
 ("B02", "B", "Estimasi Kepadatan Area Parkir Mal dari Kamera CCTV dengan YOLOv8",
  "Pengunjung sulit mengetahui ketersediaan parkir sehingga terjadi antrean kendaraan di pintu masuk.",
  "Deteksi kendaraan dengan YOLOv8 lalu penentuan tingkat kepadatan dengan ambang batas.",
  "Rekaman CCTV area parkir sebuah mal di Surabaya.",
  "klasifikasi kemacetan lalu lintas secara real", "Inspirasi DSA 2025 (kemacetan dari CCTV, YOLOv11). Objek dan dataset beda, metode serumpun"),
 ("B03", "B", "Deteksi Indikasi Depresi pada Unggahan Reddit Berbahasa Inggris dengan DistilBERT",
  "Isu kesehatan mental di media sosial meningkat dan sulit dipantau secara manual.",
  "Klasifikasi teks dengan fine-tuning DistilBERT.",
  "Unggahan subreddit depresi dan percakapan umum dari Kaggle.",
  "analisis performa model berbasis transformer  | perbandingan metode optimasi hyperparameter p | penerapan machine learning untuk deteksi kein",
  "Inspirasi DSA 2026 (x2) dan SIB 2024 (deteksi keinginan bunuh diri dari teks). Kondisi mental beda, dataset dan metode beda"),
 ("B04", "B", "Pengenalan Akor Gitar Otomatis dari Rekaman Audio dengan CNN",
  "Pemula yang belajar gitar butuh petunjuk akor sebuah lagu tetapi sulit menentukannya hanya dengan telinga.",
  "CNN pada spektrogram chroma untuk klasifikasi akor.",
  "Rekaman lagu pop Indonesia berlabel akor yang dikumpulkan sendiri.",
  "transkripsi musik otomatis pada musik polypho", "Inspirasi INFOR 2023 (transkripsi musik polifonik). Tujuan, dataset, dan metode beda"),
 ("B05", "B", "Prediksi Jumlah Donor Darah Bulanan di PMI Kota Malang dengan Prophet",
  "Persediaan kantong darah sering tidak seimbang dengan permintaan rumah sakit sehingga perlu diprediksi.",
  "Peramalan deret waktu dengan Facebook Prophet.",
  "Data donor dan permintaan darah PMI Kota Malang periode 2019 sampai 2024.",
  "prediksi kebutuhan darah menggunakan metode a | penerapan artificial neural network dan rule  | aplikasi metode random forest dan recency, fr",
  "Inspirasi SIB 2022 (prediksi kebutuhan darah ARIMA) dan dua skripsi PMI lain. Lokasi dan metode beda"),
 ("B06", "B", "Penentuan Lokasi Penyimpanan Barang Gudang Toko Online dengan Algoritma Genetika",
  "Penempatan barang yang tidak teratur membuat pengambilan pesanan di gudang memakan waktu lama.",
  "Algoritma genetika untuk penempatan lokasi barang (slotting).",
  "Data pesanan dan tata letak gudang sebuah toko online.",
  "sistem otomasi rute order picking pada gudan", "Inspirasi SIB 2022 (rute order picking, simulated annealing). Objek dan metode beda"),
 ("B07", "B", "Sistem Penilaian Otomatis Tugas Pemrograman Python pada Mata Kuliah Algoritma",
  "Dosen menghabiskan waktu lama memeriksa tugas pemrograman mahasiswa satu per satu.",
  "Pengujian otomatis dengan unit test dan autograder berbasis Docker.",
  "Kumpulan jawaban tugas pemrograman mahasiswa mata kuliah algoritma dan pemrograman.",
  "sistem penilaian otomatis untuk praktikum dig", "Inspirasi DSA 2026 (penilaian otomatis praktikum). Mata kuliah dan metode beda"),
 ("B08", "B", "Chatbot Layanan Informasi Administrasi Akademik Berbasis Retrieval-Augmented Generation",
  "Mahasiswa sering menanyakan hal yang sama kepada bagian administrasi sehingga staf kewalahan.",
  "RAG dengan embedding SBERT dan model bahasa besar untuk menjawab pertanyaan.",
  "Dokumen peraturan akademik dan FAQ universitas.",
  "sistem chatbot dengan metode naive bayes unt | klasifikasi intent pada chatbot sebagai pendu",
  "Inspirasi INFOR 2023 (chatbot tanya jawab mahasiswa pra skripsi, dan chatbot intent). Topik dan metode beda"),
 ("B09", "B", "Aplikasi Mobile Belajar Bahasa Inggris untuk Siswa SMP dengan Gamifikasi",
  "Siswa SMP cepat bosan belajar kosakata bahasa Inggris dengan cara konvensional.",
  "Game-based learning dengan poin dan level, dibangun dengan Flutter.",
  "Bank soal kosakata dan tata bahasa dari buku pelajaran SMP.",
  "aplikasi mobile persiapan ujian untuk mahasi", "Inspirasi DSA 2024 (aplikasi mobile game-based learning). Jenjang dan konten beda"),
 ("B10", "B", "Penilaian Esai Otomatis Bahasa Indonesia untuk Ujian Mahasiswa dengan IndoBERT",
  "Menilai esai secara manual memakan waktu dan hasilnya bisa tidak konsisten antar penilai.",
  "Regresi skor esai dengan fine-tuning IndoBERT.",
  "Kumpulan esai jawaban ujian mahasiswa berbahasa Indonesia beserta nilai dari dosen.",
  "penerapan gated recurrent unit dan infersent", "Inspirasi INFOR 2023 (penilaian esai otomatis, GRU). Bahasa, dataset, dan metode beda"),
 # ===== C. 10 proposal orisinal (tidak ada di sheet) =====
 ("C01", "C", "Deteksi Retakan Jalan Aspal dari Citra Drone dengan Mask R-CNN",
  "Survei kerusakan jalan secara manual lambat dan berbahaya bagi petugas dinas pekerjaan umum.",
  "Segmentasi instance dengan Mask R-CNN.", "Foto udara jalan kota dari drone dengan label piksel retakan.", "", "Orisinal"),
 ("C02", "C", "Klasifikasi Penyakit Daun Tomat dengan MobileNetV3 pada Aplikasi Mobile",
  "Petani tomat sulit membedakan jenis penyakit daun di lapangan sehingga terlambat menangani.",
  "Transfer learning MobileNetV3 dengan TensorFlow Lite.", "Foto daun tomat sehat dan berpenyakit dari PlantVillage.", "", "Orisinal (metode umum)"),
 ("C03", "C", "Penjadwalan Shift Perawat Rumah Sakit dengan Algoritma Genetika",
  "Penyusunan jadwal shift perawat secara manual sering bentrok dan terasa tidak adil.",
  "Algoritma genetika dengan batasan beban kerja.", "Data ketersediaan dan beban kerja perawat di sebuah rumah sakit tipe C.", "", "Orisinal"),
 ("C04", "C", "Deteksi Transaksi Kartu Kredit Mencurigakan dengan Autoencoder",
  "Bank terlambat mengetahui transaksi kartu kredit yang mencurigakan sehingga nasabah dirugikan.",
  "Deteksi anomali dengan autoencoder.", "Data transaksi kartu kredit anonim dari Kaggle.", "", "Orisinal"),
 ("C05", "C", "Prediksi Konsentrasi PM2.5 Harian di Surabaya dengan Random Forest",
  "Kualitas udara kota memburuk dan warga tidak mendapat peringatan dini.",
  "Random forest regression.", "Data sensor kualitas udara dan data cuaca BMKG Surabaya.", "", "Orisinal (metode umum)"),
 ("C06", "C", "Prediksi Tinggi Gelombang Laut di Selat Madura dengan LSTM",
  "Nelayan kecil sulit mendapat prakiraan gelombang yang akurat di perairan lokal.",
  "LSTM untuk peramalan deret waktu.", "Data tinggi gelombang dari satelit altimetri dan angin permukaan ERA5.", "", "Orisinal (metode umum)"),
 ("C07", "C", "Aplikasi Augmented Reality Penunjuk Jalur Evakuasi Gempa di Gedung Kampus",
  "Penghuni gedung kebingungan mencari jalur evakuasi saat terjadi gempa.",
  "Augmented reality dengan ARCore dan Unity.", "Denah gedung dan titik kumpul kampus.", "", "Orisinal"),
 ("C08", "C", "Analisis Kepuasan Wisatawan terhadap Destinasi Wisata Bromo dengan Importance-Performance Analysis",
  "Pengelola wisata belum tahu aspek layanan mana yang paling perlu diperbaiki.",
  "Importance-Performance Analysis.", "Kuesioner kepada 250 wisatawan yang berkunjung ke Bromo.", "", "Orisinal"),
 ("C09", "C", "Deteksi Dini Titik Kebakaran Hutan dari Citra Satelit dengan U-Net",
  "Kebakaran hutan sering terlambat diketahui karena areanya luas dan sulit dijangkau.",
  "Segmentasi dengan arsitektur U-Net.", "Citra Sentinel-2 dan data titik panas NASA FIRMS di Kalimantan.", "", "Orisinal (metode umum)"),
 ("C10", "C", "Prediksi Lama Rawat Inap Pasien Demam Berdarah Dengue dengan Gradient Boosting",
  "Rumah sakit sulit merencanakan ketersediaan tempat tidur karena lama rawat pasien DBD bervariasi.",
  "Gradient boosting regression.", "Data rekam medis pasien DBD sebuah rumah sakit tahun 2021 sampai 2024.", "", "Orisinal (metode umum)"),
]
df_eval = pd.DataFrame(DATA, columns=["id", "grup", "judul", "masalah", "metode", "dataset", "sumber_topik", "catatan"])
df_eval.to_csv("/content/dataset_evaluasi.csv", index=False)      # arsip dataset
df_eval[["id", "grup", "judul", "sumber_topik", "catatan"]]

,id,grup,judul,sumber_topik,catatan
0,A01,A,Aplikasi Web Belajar Aksara Sunda dengan Penge...,aplikasi pembelajaran dan pengenalan aksara s,"Parafrase DSA 2025 (aksara Sunda, EfficientNet..."
1,A02,A,Peningkatan Akurasi Pengenalan Emosi dari Suar...,integrasi teks dan suara dengan model wav2vec,Parafrase DSA 2025 (speech emotion recognition)
2,A03,A,Faktor yang Memengaruhi Indeks Pembangunan Man...,analisis faktor-faktor yang mempengaruhi inde,Parafrase DSA 2026 (IPM Jawa Timur)
3,A04,A,Penentuan Lokasi Terbaik Pembangkit Listrik Te...,analisis data spasio-temporal untuk identifik,Parafrase DSA 2026 (lokasi optimal PLTB NTT)
4,A05,A,Pembacaan Gerak Bibir Bahasa Indonesia Berdasa...,pemodelan lip reading bahasa indonesia berbas,Parafrase INFOR 2022 (lip reading visem)
5,A06,A,Kendali Presentasi dan Pemutar Media dengan Ge...,deteksi hand gesture dengan menggunakan media,Parafrase INFOR 2022 (hand gesture Mediapipe)
6,A07,A,Aplikasi Web Penerbitan Sertifikat dengan Peme...,aplikasi pembuatan sertifikat dengan validasi,Parafrase INFOR 2023 (sertifikat SHA-256 RSA)
7,A08,A,Gudang Data Berbasis Graph Database untuk Meng...,data warehouse dengan graph database untuk an,Parafrase SIB 2024 (data warehouse graph datab...
8,A09,A,Optimasi Rute Pengantaran pada Jasa Kurir deng...,sistem optimalisasi rute menggunakan metode c,Parafrase SIB 2023 (rute Clarke and Wright)
9,A10,A,Aplikasi Pendamping Konsultasi Diet Online unt...,aplikasi konsultasi diet jp smart fitness,Parafrase SIB 2023 (konsultasi diet JP Smart F...


In [ ]:
def evaluasi(mesin, data, k=5):
    """Jalankan semua proposal, lalu nilai lulus/tidak sesuai aturan tiap kelompok."""
    norm = lambda t: re.sub(r"\s+", " ", t).strip().strip('"').lower()
    judul_db = [norm(t) for t in mesin.d.judul]
    baris = []
    for id_, grup, judul, masalah, metode, dataset, sumber, catatan in data:
        s = mesin.skor(judul, masalah, metode, dataset)
        top = int(np.argmax(s))
        awalan = [p.strip() for p in sumber.split(" | ") if p.strip()]
        idx = [i for i, j in enumerate(judul_db) if any(j.startswith(p) for p in awalan)]
        rank = int(min((s > s[i]).sum() + 1 for i in idx)) if idx else None   # peringkat skripsi sumber (1 = teratas)
        tingkat = mesin.tingkat(s[top])
        lulus = {"A": rank == 1 and tingkat != "RENDAH",       # A: sumber peringkat 1 dan terdeteksi mirip
                 "B": rank is not None and rank <= k,         # B: sumber masuk top-k
                 "C": tingkat == "RENDAH"}[grup]              # C: tidak ada yang dianggap mirip
        baris.append(dict(id=id_, grup=grup, skor_top1=round(s[top], 3), tingkat=tingkat,
                          peringkat_sumber=rank, top1=mesin.d.judul[top][:55], lulus=lulus))
    return pd.DataFrame(baris)

r = evaluasi(mesin, DATA)
display(r)                                                                   # hasil per proposal
display(r.groupby("grup").agg(jumlah=("id", "count"), lulus=("lulus", "sum"), skor_rata2=("skor_top1", "mean"),
                              skor_min=("skor_top1", "min"), skor_maks=("skor_top1", "max")).round(3))
display(pd.crosstab(r.grup, r.tingkat))                                      # sebaran tingkat per kelompok
for a in (1.0, 0.7, 0.0):                                                    # 1.0 = makna saja, 0.0 = kata kunci saja
    mesin.alpha = a
    print(f"alpha={a}:", evaluasi(mesin, DATA).groupby("grup").lulus.sum().to_dict())
mesin.alpha = 0.7

,id,grup,skor_top1,tingkat,peringkat_sumber,top1,lulus
0,A01,A,0.585,RENDAH,1.0,Aplikasi pembelajaran dan pengenalan aksara Su...,False
1,A02,A,0.743,MIRIP,1.0,Integrasi teks dan suara dengan model wav2vec ...,True
2,A03,A,0.601,MIRIP,1.0,Analisis faktor-faktor yang mempengaruhi Indek...,True
3,A04,A,0.670,MIRIP,1.0,Analisis data spasio-temporal untuk identifika...,True
4,A05,A,0.697,MIRIP,1.0,Pemodelan lip reading bahasa Indonesia berbasi...,True
5,A06,A,0.514,RENDAH,1.0,Deteksi hand gesture dengan menggunakan Mediap...,False
6,A07,A,0.693,MIRIP,1.0,Aplikasi pembuatan sertifikat dengan validasi ...,True
7,A08,A,0.787,SANGAT MIRIP,1.0,Data warehouse dengan graph database untuk ana...,True
8,A09,A,0.735,MIRIP,1.0,Sistem optimalisasi rute menggunakan metode cl...,True
9,A10,A,0.633,MIRIP,1.0,Aplikasi konsultasi diet JP Smart Fitness,True


,jumlah,lulus,skor_rata2,skor_min,skor_maks
grup,,,,,
A,10,8,0.666,0.514,0.787
B,10,9,0.502,0.409,0.602
C,10,10,0.378,0.303,0.514


tingkat,MIRIP,RENDAH,SANGAT MIRIP
grup,,,
A,7,2,1
B,1,9,0
C,0,10,0


alpha=1.0: {'A': 6, 'B': 10, 'C': 9}
alpha=0.7: {'A': 8, 'B': 9, 'C': 10}
alpha=0.0: {'A': 7, 'B': 7, 'C': 10}
